# 02 - Transformação Silver
Faz o parse do layout posicional COTAHIST, aplica tipos, filtros, regras básicas e deduplicação.

In [0]:
%run ./00_setup

# 00 - Configuração
Cria objetos do Unity Catalog e caminhos do MVP. Ajuste os widgets antes da primeira execução.

Envie COTAHIST_A2025.TXT para: /Volumes/workspace/mvp_b3/landing/cotahist/
Envie setores_b3.csv para: /Volumes/workspace/mvp_b3/landing/referencia/


In [0]:
from pyspark.sql import functions as F, Window

src = spark.table(f"{catalog}.{schema}.bronze_cotahist_raw")

def txt(start, length):
    return F.trim(F.substring("raw_line", start, length))

def money(start, length):
    return (txt(start, length).cast("decimal(24,0)") / F.lit(100)).cast("decimal(24,2)")

parsed = src.select(
    F.to_date(txt(3, 8), "yyyyMMdd").alias("data_pregao"),
    txt(11, 2).alias("cod_bdi"),
    F.upper(txt(13, 12)).alias("ticker"),
    txt(25, 3).cast("int").alias("tipo_mercado"),
    txt(28, 12).alias("nome_resumido"),
    txt(40, 10).alias("especificacao"),
    txt(50, 3).alias("prazo_termo"),
    txt(53, 4).alias("moeda"),
    money(57, 13).alias("preco_abertura"),
    money(70, 13).alias("preco_maximo"),
    money(83, 13).alias("preco_minimo"),
    money(96, 13).alias("preco_medio"),
    money(109, 13).alias("preco_fechamento"),
    money(122, 13).alias("preco_melhor_compra"),
    money(135, 13).alias("preco_melhor_venda"),
    txt(148, 5).cast("long").alias("numero_negocios"),
    txt(153, 18).cast("long").alias("quantidade_titulos"),
    money(171, 18).alias("volume_financeiro"),
    txt(211, 7).cast("long").alias("fator_cotacao"),
    txt(231, 12).alias("isin"),
    "source_file", "ingestion_ts"
)

# Mercado à vista (010) e papéis ON/PN; removes linhas inválidas antes da deduplicação.
valid = parsed.filter(
    (F.col("tipo_mercado") == 10) &
    (F.col("ticker") != "") &
    (F.col("data_pregao").isNotNull()) &
    (F.col("preco_fechamento") > 0) &
    (F.col("preco_maximo") >= F.col("preco_minimo")) &
    (F.col("volume_financeiro") >= 0) &
    (F.col("especificacao").rlike("^(ON|PN)"))
)

w = Window.partitionBy("data_pregao", "ticker").orderBy(F.col("ingestion_ts").desc())
silver = valid.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

(silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{catalog}.{schema}.silver_cotacoes"))

empresas = (spark.table(f"{catalog}.{schema}.bronze_setores_raw")
    .select(F.upper(F.trim("ticker")).alias("ticker"),
            F.trim("empresa").alias("empresa"),
            F.coalesce(F.upper(F.trim("setor")), F.lit("NAO_INFORMADO")).alias("setor"),
            F.coalesce(F.upper(F.trim("subsetor")), F.lit("NAO_INFORMADO")).alias("subsetor"),
            F.coalesce(F.upper(F.trim("segmento")), F.lit("NAO_INFORMADO")).alias("segmento"))
    .filter("ticker IS NOT NULL AND ticker <> ''").dropDuplicates(["ticker"]))

(empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{catalog}.{schema}.silver_empresas"))

display(silver.groupBy(F.year("data_pregao").alias("ano")).agg(F.count("*").alias("registros"), F.countDistinct("ticker").alias("ativos")).orderBy("ano"))


ano,registros,ativos
2025,85890,457
